# GEWS PoC — Nepal 2026 Glacier Collapse Analysis

**Global Early Warning System — InSAR Pipeline Proof of Concept**

This notebook runs the full GEWS pipeline on real Sentinel-1 data covering the
26 August 2026 Nepal–Tibet border glacier–rock collapse, using ASF's cloud
infrastructure:

1. **Search** for Sentinel-1 burst SLCs over the collapse site
2. **Submit** interferogram jobs to HyP3 (ASF's cloud processing)
3. **Download** processed interferograms
4. **Run MintPy** time-series inversion → displacement time series
5. **Run GEWS** anomaly detection → acceleration z-score maps
6. **Report** — did the pipeline detect the pre-collapse signal?

**Environment:** Designed for [ASF OpenSARLab](https://opensciencelab.asf.alaska.edu/).
Can also run locally with `hyp3_sdk`, `mintpy`, and `gews` installed.

---

## 0. Setup and Configuration

In [ ]:
# Install GEWS if not already available
# In OpenSARLab, upload the gews source directory first, then:
# !pip install -e /path/to/gews
#
# Or install just the core dependencies (numpy, scipy, scikit-learn
# are already in OpenSARLab):
# !pip install ruptures

import sys
from pathlib import Path

# Add gews source to path if installed locally
GEWS_SRC = Path("../src").resolve()
if GEWS_SRC.exists() and str(GEWS_SRC) not in sys.path:
    sys.path.insert(0, str(GEWS_SRC))

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Site parameters
SITE_LAT = 28.20          # Approximate collapse source latitude
SITE_LON = 85.90          # Approximate collapse source longitude
BUFFER_KM = 15            # Search radius
EVENT_DATE = "2026-08-26" # Collapse date

# Time range for analysis
START_DATE = "2025-01-01"  # 20 months of baseline
END_DATE = "2026-08-26"   # Up to and including collapse

# Track selection (from our search: Track 19 has data through Aug 24)
RELATIVE_ORBIT = 19

# Processing parameters
PROJECT_NAME = "gews_nepal_2026"  # HyP3 project name
POLARIZATION = "VV"

# Working directories
DATA_DIR = Path("data")
HYP3_DIR = DATA_DIR / "hyp3_products"
MINT_DIR = DATA_DIR / "mintpy"
OUTPUT_DIR = Path("output")

for d in [DATA_DIR, HYP3_DIR, MINT_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Site: {SITE_LAT}°N, {SITE_LON}°E")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Relative orbit: {RELATIVE_ORBIT}")

---
## 1. Search for Sentinel-1 Burst SLCs

We search ASF for burst-level SLC granules covering the study area.
HyP3's INSAR_ISCE_BURST job type operates on individual bursts rather
than full SLC frames — this is more efficient and avoids downloading
the full 4+ GB SLC files.

In [ ]:
import asf_search as asf

# Search for burst SLCs covering the study area
search_results = asf.search(
    platform=asf.PLATFORM.SENTINEL1,
    processingLevel=asf.PRODUCT_TYPE.BURST,
    beamMode=asf.BEAMMODE.IW,
    intersectsWith=f"POINT({SITE_LON} {SITE_LAT})",
    relativeOrbit=[RELATIVE_ORBIT],
    polarization=["VV"],
    start=START_DATE,
    end=END_DATE,
)

print(f"Found {len(search_results)} burst SLC granules on track {RELATIVE_ORBIT}")

# Group by burst ID to understand spatial coverage
burst_ids = {}
for r in search_results:
    bid = r.properties.get("burst", {}).get("relativeBurstID", 
          r.properties.get("fileID", "").split("_")[1] if "BURST" in r.properties.get("fileID", "") else "unknown")
    burst_ids.setdefault(bid, []).append(r)

print(f"Covering {len(burst_ids)} unique burst positions")
print()
for bid, bursts in sorted(burst_ids.items(), key=lambda x: len(x[1]), reverse=True)[:5]:
    dates = sorted(set(r.properties["startTime"][:10] for r in bursts))
    print(f"  Burst {bid}: {len(bursts)} acquisitions, {dates[0]} to {dates[-1]}")

In [ ]:
# Select the burst with the most acquisitions (best temporal sampling)
# For a full analysis, you'd process multiple bursts to cover the
# whole study area. For the PoC, one burst covering the collapse site
# is sufficient.

best_burst_id = max(burst_ids, key=lambda k: len(burst_ids[k]))
burst_granules = sorted(burst_ids[best_burst_id], key=lambda r: r.properties["startTime"])

print(f"Selected burst: {best_burst_id}")
print(f"Granules: {len(burst_granules)}")
print(f"Date range: {burst_granules[0].properties['startTime'][:10]} to {burst_granules[-1].properties['startTime'][:10]}")
print()

# List all acquisition dates
granule_ids = [r.properties["fileID"] for r in burst_granules]
acq_dates = [r.properties["startTime"][:10] for r in burst_granules]

print("Acquisition dates:")
for i, (gid, date) in enumerate(zip(granule_ids, acq_dates)):
    marker = " ← LAST BEFORE COLLAPSE" if date == max(d for d in acq_dates if d <= EVENT_DATE) else ""
    print(f"  {i+1:3d}. {date}  {gid}{marker}")

---
## 2. Submit HyP3 Interferogram Jobs

We submit burst InSAR jobs to HyP3 using a sequential network:
each scene is paired with its nearest temporal neighbors.
This creates an SBAS (Small Baseline Subset) network suitable
for MintPy time-series inversion.

**Note:** HyP3 Basic provides a monthly credit allotment.
For ~50 granules with 2 connections each, we need ~100 jobs.
This may take 1–2 months of free credits. Apply for HyP3+
research credits if available.

In [ ]:
from hyp3_sdk import HyP3

# Authenticate with Earthdata Login
# In OpenSARLab, credentials are often cached.
# Otherwise, you'll be prompted for username/password.
hyp3 = HyP3(prompt=True)

In [ ]:
def build_sbas_pairs(granule_ids, n_connections=2):
    """
    Build SBAS interferogram pairs from a sequential list of granules.
    
    Each granule is paired with its n_connections nearest temporal
    neighbors, creating a connected network for time-series inversion.
    
    Parameters
    ----------
    granule_ids : list[str]
        Burst SLC granule IDs, sorted by acquisition date.
    n_connections : int
        Number of forward connections per granule.
    
    Returns
    -------
    list[tuple[str, str]]
        (reference, secondary) granule ID pairs.
    """
    pairs = []
    for i in range(len(granule_ids)):
        for j in range(1, n_connections + 1):
            if i + j < len(granule_ids):
                pairs.append((granule_ids[i], granule_ids[i + j]))
    return pairs


# Build sequential SBAS network
pairs = build_sbas_pairs(granule_ids, n_connections=2)

print(f"SBAS network: {len(pairs)} interferogram pairs from {len(granule_ids)} granules")
print(f"Network connectivity: {2} forward connections per scene")
print()
print("First 5 pairs:")
for ref, sec in pairs[:5]:
    print(f"  {ref}  ↔  {sec}")
print("...")
print(f"Last pair: {pairs[-1][0]}  ↔  {pairs[-1][1]}")

In [ ]:
# Check for already-submitted jobs
existing_jobs = hyp3.find_jobs(
    name=PROJECT_NAME,
    job_type="INSAR_ISCE_BURST",
).filter_jobs(include_expired=False)

existing_pairs = set()
for job in existing_jobs:
    params = job.job_parameters
    granules = params.get("granules", [])
    if len(granules) == 2:
        existing_pairs.add((granules[0], granules[1]))

new_pairs = [p for p in pairs if p not in existing_pairs and (p[1], p[0]) not in existing_pairs]

print(f"Already submitted: {len(existing_pairs)} pairs")
print(f"New to submit: {len(new_pairs)} pairs")

In [ ]:
# Submit new interferogram jobs
# ⚠️ This submits real processing jobs that count against your HyP3 quota.
# Review the pairs above before running this cell.

SUBMIT_JOBS = False  # ← Set to True when ready to submit

if SUBMIT_JOBS and new_pairs:
    submitted = []
    for i, (ref, sec) in enumerate(new_pairs):
        job = hyp3.submit_insar_isce_burst_job(
            granule1=ref,
            granule2=sec,
            name=PROJECT_NAME,
            looks="20x4",  # 80m resolution — best for regional screening
        )
        submitted.append(job)
        if (i + 1) % 10 == 0:
            print(f"  Submitted {i + 1}/{len(new_pairs)} jobs...")
    
    print(f"\nSubmitted {len(submitted)} jobs as project '{PROJECT_NAME}'")
    print("Jobs will process in the cloud. Check status with the next cell.")
else:
    if not new_pairs:
        print("All pairs already submitted.")
    else:
        print(f"Set SUBMIT_JOBS = True to submit {len(new_pairs)} jobs.")
        print("Review the pairs above first.")

In [ ]:
# Check job status
batch = hyp3.find_jobs(
    name=PROJECT_NAME,
    job_type="INSAR_ISCE_BURST",
).filter_jobs(include_expired=False)

statuses = {}
for job in batch:
    s = job.status_code
    statuses[s] = statuses.get(s, 0) + 1

print(f"Project '{PROJECT_NAME}': {len(batch)} jobs")
for status, count in sorted(statuses.items()):
    print(f"  {status}: {count}")

if batch.complete():
    print("\n✓ All jobs complete. Ready for download.")
else:
    remaining = sum(v for k, v in statuses.items() if k in ("PENDING", "RUNNING"))
    print(f"\n⏳ {remaining} jobs still processing. Re-run this cell to check.")

---
## 3. Download Processed Interferograms

Once HyP3 jobs are complete, download the products.
In OpenSARLab, this is fast — the data stays within AWS.

In [ ]:
from tqdm.auto import tqdm
import zipfile

# Download completed jobs
batch = hyp3.find_jobs(
    name=PROJECT_NAME,
    job_type="INSAR_ISCE_BURST",
).filter_jobs(running=False, include_expired=False)

succeeded = [job for job in batch if job.status_code == "SUCCEEDED"]
print(f"Downloading {len(succeeded)} completed products to {HYP3_DIR}")

for job in tqdm(succeeded, desc="Downloading"):
    job.download_files(HYP3_DIR)

# Unzip all products
zip_files = list(HYP3_DIR.glob("*.zip"))
print(f"\nUnzipping {len(zip_files)} products...")

for zf in tqdm(zip_files, desc="Unzipping"):
    with zipfile.ZipFile(zf, "r") as z:
        z.extractall(HYP3_DIR)

product_dirs = sorted([d for d in HYP3_DIR.iterdir() if d.is_dir()])
print(f"\n{len(product_dirs)} interferogram products ready")

---
## 4. MintPy Time-Series Inversion

We use MintPy to invert the interferogram stack into a displacement
time series. This handles:
- Network inversion (SBAS)
- Atmospheric phase screen estimation and correction
- DEM error correction
- Velocity estimation

The output is a 3D displacement cube: [n_dates, n_rows, n_cols].

In [ ]:
# Generate MintPy configuration for HyP3 burst InSAR products
#
# HyP3 INSAR_ISCE_BURST products are GeoTIFFs, which MintPy can
# load directly. The key files in each product directory are:
#   *_unw_phase.tif    — unwrapped interferogram
#   *_corr.tif         — coherence
#   *_dem.tif           — DEM
#   *_lv_theta.tif     — look vector (incidence angle)
#   *_lv_phi.tif       — look vector (azimuth angle)
#   *_conncomp.tif     — connected components

mintpy_cfg = f"""
##-------------- MintPy Configuration for GEWS PoC ----------------##
## Site: Nepal-Tibet Border 2026 Glacier Collapse
## Data: HyP3 INSAR_ISCE_BURST products, Track {RELATIVE_ORBIT}

########## loading data ##########
mintpy.load.processor        = hyp3
mintpy.load.unwFile          = {HYP3_DIR}/*/*_unw_phase.tif
mintpy.load.corFile           = {HYP3_DIR}/*/*_corr.tif
mintpy.load.connCompFile     = {HYP3_DIR}/*/*_conncomp.tif
mintpy.load.demFile           = {HYP3_DIR}/*/*_dem.tif
mintpy.load.lookupYFile      = None
mintpy.load.lookupXFile      = None
mintpy.load.incAngleFile     = {HYP3_DIR}/*/*_lv_theta.tif
mintpy.load.azAngleFile      = {HYP3_DIR}/*/*_lv_phi.tif
mintpy.load.waterMaskFile    = {HYP3_DIR}/*/*_water_mask.tif

########## reference point ##########
## Auto-select a stable reference point
mintpy.reference.lalo        = auto

########## network inversion ##########
mintpy.networkInversion.weightFunc = var
mintpy.networkInversion.minTempCoh = 0.7

########## tropospheric correction ##########
## ERA5 is the best option but requires PyAPS and an ECMWF account.
## Use height_correlation as a simpler fallback.
mintpy.troposphericDelay.method = height_correlation

########## topographic residual ##########
mintpy.topographicResidual   = yes

########## geocoding ##########
## HyP3 products are already geocoded — no re-geocoding needed
mintpy.geocode               = no
"""

config_path = MINT_DIR / "smallbaselineApp.cfg"
config_path.write_text(mintpy_cfg)
print(f"MintPy config written to {config_path}")
print()
print(mintpy_cfg)

In [ ]:
# Run MintPy — step by step for visibility
#
# Each step takes a few seconds to minutes depending on stack size.
# If a step fails, you can re-run from that step onward.

import mintpy.cli.smallbaselineApp as sba

MINTPY_STEPS = [
    "load_data",
    "modify_network",
    "reference_point",
    "quick_overview",
    "correct_unwrap_error",
    "invert_network",
    "correct_LOD",
    "correct_SET",
    "correct_troposphere",
    "deramp",
    "correct_topography",
    "residual_RMS",
    "reference_date",
    "velocity",
    "geocode",
    "google_earth",
    "hdfeos5",
]

START_FROM_STEP = "load_data"  # Change this to resume from a specific step

run_steps = MINTPY_STEPS[MINTPY_STEPS.index(START_FROM_STEP):]
print(f"Running MintPy steps: {', '.join(run_steps)}")
print()

for step in run_steps:
    print(f"--- Step: {step} ---")
    try:
        args = f"{config_path} --work-dir={MINT_DIR} --dostep {step}"
        sba.main(args.split())
        print(f"  ✓ {step} complete")
    except SystemExit:
        print(f"  ✓ {step} complete (SystemExit is normal for MintPy)")
    except Exception as e:
        print(f"  ✗ {step} failed: {e}")
        print(f"    Fix the issue and set START_FROM_STEP = '{step}' to retry")
        break

print("\nMintPy processing complete.")

In [ ]:
# Verify MintPy outputs
expected_files = [
    "timeseries.h5",
    "temporalCoherence.h5",
    "velocity.h5",
    "maskTempCoh.h5",
    "geometryGeo.h5",    # or geometryRadar.h5 depending on geocoding
]

print("MintPy output files:")
for f in sorted(MINT_DIR.glob("*.h5")):
    size_mb = f.stat().st_size / 1e6
    marker = " ✓" if f.name in expected_files else ""
    print(f"  {f.name:40s} {size_mb:8.1f} MB{marker}")

---
## 5. Load Time Series into GEWS Format

Convert MintPy's HDF5 output into the `DisplacementTimeseries`
format that our anomaly detection pipeline expects.

In [ ]:
import h5py
from mintpy.utils import readfile


def load_mintpy_timeseries(mint_dir, min_coherence=0.7):
    """
    Load MintPy output into numpy arrays for GEWS processing.
    
    Returns dates, displacement, velocity, coherence, lat, lon.
    """
    mint_dir = Path(mint_dir)
    
    # Find the best time-series file (with corrections applied)
    ts_candidates = [
        "timeseries_ERA5_ramp_demErr.h5",
        "timeseries_tropHgt_ramp_demErr.h5",
        "timeseries_demErr.h5",
        "timeseries.h5",
    ]
    
    ts_file = None
    for candidate in ts_candidates:
        if (mint_dir / candidate).exists():
            ts_file = mint_dir / candidate
            break
    
    if ts_file is None:
        raise FileNotFoundError(f"No time-series file found in {mint_dir}")
    
    print(f"Loading time series from: {ts_file.name}")
    
    # Load displacement time series
    with h5py.File(ts_file, "r") as f:
        displacement = f["timeseries"][:]  # [n_dates, n_rows, n_cols] in meters
        date_strings = [d.decode() for d in f["date"][:]]
    
    # Load velocity
    with h5py.File(mint_dir / "velocity.h5", "r") as f:
        velocity = f["velocity"][:]
    
    # Load temporal coherence
    with h5py.File(mint_dir / "temporalCoherence.h5", "r") as f:
        coherence = f["temporalCoherence"][:]
    
    # Load geometry (coordinates)
    geom_file = mint_dir / "geometryGeo.h5"
    if not geom_file.exists():
        geom_file = mint_dir / "geometryRadar.h5"
    
    # For HyP3 geocoded products, coordinates come from the GeoTIFF
    # metadata. Read from the attributes of the time-series file.
    atr = readfile.read_attribute(str(ts_file))
    n_rows, n_cols = displacement.shape[1:]
    
    # Build coordinate grids from metadata
    y_first = float(atr.get("Y_FIRST", SITE_LAT + 0.1))
    x_first = float(atr.get("X_FIRST", SITE_LON - 0.1))
    y_step = float(atr.get("Y_STEP", -0.001))
    x_step = float(atr.get("X_STEP", 0.001))
    
    lat_1d = y_first + np.arange(n_rows) * y_step
    lon_1d = x_first + np.arange(n_cols) * x_step
    longitude, latitude = np.meshgrid(lon_1d, lat_1d)
    # meshgrid with (lon_1d, lat_1d) gives shape [n_rows, n_cols]
    # where latitude[r, c] varies with r and longitude[r, c] varies with c
    
    # Parse dates to ordinal
    dates = np.array([
        datetime.strptime(d, "%Y%m%d").toordinal() for d in date_strings
    ])
    
    # Apply coherence mask
    mask = coherence < min_coherence
    displacement[:, mask] = np.nan
    velocity[mask] = np.nan
    
    n_valid = np.count_nonzero(~mask)
    print(f"Loaded: {len(date_strings)} dates × {n_rows}×{n_cols} pixels")
    print(f"Valid pixels: {n_valid:,} ({100*n_valid/mask.size:.1f}%, coherence ≥ {min_coherence})")
    print(f"Date range: {date_strings[0]} to {date_strings[-1]}")
    print(f"Lat range: {latitude.min():.4f} to {latitude.max():.4f}")
    print(f"Lon range: {longitude.min():.4f} to {longitude.max():.4f}")
    
    return {
        "dates": dates,
        "date_strings": date_strings,
        "displacement": displacement,
        "velocity": velocity,
        "coherence": coherence,
        "latitude": latitude,
        "longitude": longitude,
        "metadata": atr,
    }


ts_data = load_mintpy_timeseries(MINT_DIR)

In [ ]:
# Quick look at MintPy velocity map
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Velocity
vmax = np.nanpercentile(np.abs(ts_data["velocity"]) * 1000, 95)
im = axes[0].pcolormesh(
    ts_data["longitude"], ts_data["latitude"],
    ts_data["velocity"] * 1000,  # convert to mm/yr
    cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto",
)
plt.colorbar(im, ax=axes[0], label="LOS Velocity (mm/yr)")
axes[0].plot(SITE_LON, SITE_LAT, "k*", markersize=15, label="Collapse site")
axes[0].set_title("MintPy LOS Velocity")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
axes[0].legend()

# Temporal coherence
im2 = axes[1].pcolormesh(
    ts_data["longitude"], ts_data["latitude"],
    ts_data["coherence"],
    cmap="gray", vmin=0, vmax=1, shading="auto",
)
plt.colorbar(im2, ax=axes[1], label="Temporal Coherence")
axes[1].plot(SITE_LON, SITE_LAT, "r*", markersize=15)
axes[1].set_title("Temporal Coherence")
axes[1].set_xlabel("Longitude")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "mintpy_velocity.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {OUTPUT_DIR / 'mintpy_velocity.png'}")

---
## 6. GEWS Anomaly Detection (Tier 0)

This is the core of the PoC: run our acceleration-based anomaly
detection on the real displacement time series.

For each pixel, the algorithm:
1. Removes seasonal signal (annual + semi-annual harmonics)
2. Computes sliding-window acceleration
3. Normalizes against each pixel's own baseline (z-score)
4. Clusters spatially connected anomalous pixels
5. Scores and ranks clusters

In [ ]:
from gews.timeseries import compute_acceleration_map
from gews.detect import detect_anomalies

# Detection configuration — same parameters as the proposal
detect_config = {
    "detect": {
        "n_harmonics": 2,
        "acceleration": {
            "window_size_days": 60,
            "step_days": 12,
            "sigma_threshold": 2.5,
            "changepoint": {
                "model": "rbf",
                "penalty": "bic",
                "min_segment_size": 3,
            },
        },
        "clustering": {
            "min_cluster_pixels": 5,
            "max_distance_m": 200,
            "min_area_m2": 10000,
        },
        "voight": {
            "enabled": True,
            "min_points": 5,
            "r_squared_threshold": 0.7,
        },
    }
}

print("Computing acceleration map...")
accel_map = compute_acceleration_map(
    ts_data["dates"],
    ts_data["displacement"],
    window_size_days=60,
    step_days=12,
    n_harmonics=2,
)

print(f"Acceleration map: {accel_map.acceleration_zscore.shape[0]} windows × "
      f"{accel_map.acceleration_zscore.shape[1]}×{accel_map.acceleration_zscore.shape[2]} pixels")
print(f"Max |z-score|: {np.nanmax(np.abs(accel_map.acceleration_zscore)):.1f}")
print()

print("Running anomaly detection...")
flags = detect_anomalies(
    accel_map,
    ts_data["latitude"],
    ts_data["longitude"],
    detect_config,
)

print(f"\nDetected {len(flags)} anomaly flags")
print()

if flags:
    print(f"{'Flag':>6} {'Score':>7} {'Peak Z':>8} {'Area (m²)':>12} {'Accel (mm/yr²)':>16} {'Location':>30}")
    print("-" * 85)
    for f in flags[:15]:
        loc = f"{f.center_lat:.4f}°N, {f.center_lon:.4f}°E"
        print(f"{f.flag_id:>6d} {f.score:>7.2f} {f.peak_zscore:>8.1f} "
              f"{f.area_m2:>12,.0f} {f.acceleration_m_yr2*1000:>16.2f} {loc:>30}")

---
## 7. Results Visualization

In [ ]:
from matplotlib.colors import TwoSlopeNorm

# Acceleration anomaly map — latest window (closest to collapse)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Latest z-score map
latest_z = accel_map.acceleration_zscore[-1]
vmax = max(3, np.nanpercentile(np.abs(latest_z), 99))
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

im = axes[0].pcolormesh(
    ts_data["longitude"], ts_data["latitude"], latest_z,
    cmap="RdBu_r", norm=norm, shading="auto",
)
plt.colorbar(im, ax=axes[0], label="Acceleration z-score", shrink=0.8)

# Mark flagged sites
for f in flags[:20]:
    axes[0].plot(f.center_lon, f.center_lat, "k^", markersize=8,
                 markeredgewidth=1.5, markerfacecolor="none")

# Mark known collapse site
axes[0].plot(SITE_LON, SITE_LAT, "r*", markersize=18, label="Collapse site",
             markeredgecolor="darkred", markeredgewidth=0.5)

last_date = datetime.fromordinal(int(accel_map.window_centers[-1])).strftime("%Y-%m-%d")
axes[0].set_title(f"Acceleration Anomaly — Window ending {last_date}")
axes[0].set_xlabel("Longitude (°E)")
axes[0].set_ylabel("Latitude (°N)")
axes[0].legend(loc="upper right")
axes[0].set_aspect("equal")

# Second-to-last window for comparison
if accel_map.acceleration_zscore.shape[0] > 5:
    earlier_z = accel_map.acceleration_zscore[-6]  # ~2 months earlier
    im2 = axes[1].pcolormesh(
        ts_data["longitude"], ts_data["latitude"], earlier_z,
        cmap="RdBu_r", norm=norm, shading="auto",
    )
    plt.colorbar(im2, ax=axes[1], label="Acceleration z-score", shrink=0.8)
    axes[1].plot(SITE_LON, SITE_LAT, "r*", markersize=18,
                 markeredgecolor="darkred", markeredgewidth=0.5)
    earlier_date = datetime.fromordinal(int(accel_map.window_centers[-6])).strftime("%Y-%m-%d")
    axes[1].set_title(f"Acceleration Anomaly — Window ending {earlier_date}")
    axes[1].set_xlabel("Longitude (°E)")
    axes[1].set_aspect("equal")

fig.suptitle("GEWS Tier 0 — Nepal-Tibet Border 2026", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "anomaly_map_real.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Time series at the collapse site
# Find the nearest pixel to the known collapse location

lat = ts_data["latitude"]
lon = ts_data["longitude"]
dist = (lat - SITE_LAT)**2 + (lon - SITE_LON)**2
site_row, site_col = np.unravel_index(np.nanargmin(dist), dist.shape)

print(f"Nearest pixel to collapse site: row={site_row}, col={site_col}")
print(f"  Coordinates: {lat[site_row, site_col]:.4f}°N, {lon[site_row, site_col]:.4f}°E")
print(f"  Temporal coherence: {ts_data['coherence'][site_row, site_col]:.3f}")
print(f"  Mean velocity: {ts_data['velocity'][site_row, site_col]*1000:.2f} mm/yr")

# Plot displacement time series at this pixel
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

dates_dt = [datetime.fromordinal(int(d)) for d in ts_data["dates"]]
disp_mm = ts_data["displacement"][:, site_row, site_col] * 1000  # to mm

# Displacement
axes[0].plot(dates_dt, disp_mm, "b-o", markersize=4, linewidth=1)
axes[0].set_ylabel("LOS Displacement (mm)")
axes[0].set_title(f"Displacement at collapse site ({lat[site_row, site_col]:.4f}°N, {lon[site_row, site_col]:.4f}°E)")
axes[0].axvline(datetime.strptime(EVENT_DATE, "%Y-%m-%d"), color="red",
                linestyle="--", alpha=0.7, label="Collapse date")
axes[0].legend()

# Acceleration z-score at this pixel
window_dates = [datetime.fromordinal(int(d)) for d in accel_map.window_centers]
z_at_site = accel_map.acceleration_zscore[:, site_row, site_col]

axes[1].fill_between(window_dates, z_at_site, 0, alpha=0.3, color="steelblue")
axes[1].plot(window_dates, z_at_site, "b-o", markersize=3, linewidth=1)
axes[1].axhline(2.5, color="red", linestyle="--", label="Threshold (σ=2.5)")
axes[1].axhline(-2.5, color="red", linestyle="--")
axes[1].axvline(datetime.strptime(EVENT_DATE, "%Y-%m-%d"), color="red",
                linestyle="--", alpha=0.7)
axes[1].set_ylabel("Acceleration z-score")
axes[1].set_title("Acceleration Anomaly (vs. site baseline)")
axes[1].legend()

# Velocity evolution (sliding window)
vel_at_site = accel_map.velocity_residual[:, site_row, site_col] * 1000  # mm/yr
axes[2].plot(window_dates, vel_at_site, "g-o", markersize=3, linewidth=1)
axes[2].axvline(datetime.strptime(EVENT_DATE, "%Y-%m-%d"), color="red",
                linestyle="--", alpha=0.7)
axes[2].set_ylabel("Residual Velocity (mm/yr)")
axes[2].set_xlabel("Date")
axes[2].set_title("Detrended, Deseasoned Velocity")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "collapse_site_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nSaved to {OUTPUT_DIR / 'collapse_site_timeseries.png'}")

In [ ]:
# Generate GeoJSON of flagged sites for viewing in Google Earth / QGIS
import json

features = []
for f in flags:
    features.append({
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [float(f.center_lon), float(f.center_lat)],
        },
        "properties": {
            "flag_id": int(f.flag_id),
            "score": round(float(f.score), 2),
            "peak_zscore": round(float(f.peak_zscore), 2),
            "area_m2": round(float(f.area_m2), 0),
            "acceleration_mm_yr2": round(float(f.acceleration_m_yr2 * 1000), 2),
        },
    })

# Add the known collapse site
features.append({
    "type": "Feature",
    "geometry": {"type": "Point", "coordinates": [SITE_LON, SITE_LAT]},
    "properties": {"name": "Known collapse site", "date": EVENT_DATE},
})

geojson = {
    "type": "FeatureCollection",
    "features": features,
}

geojson_path = OUTPUT_DIR / "nepal_2026_flags.geojson"
geojson_path.write_text(json.dumps(geojson, indent=2))
print(f"GeoJSON with {len(flags)} flags saved to {geojson_path}")
print("Open in QGIS or Google Earth to overlay on imagery.")

---
## 8. Validation Summary

The key question: **did the pipeline detect anomalous acceleration
near the collapse site before the event?**

In [ ]:
# Find the flag nearest to the known collapse site
if flags:
    distances = [
        np.sqrt((f.center_lat - SITE_LAT)**2 + (f.center_lon - SITE_LON)**2)
        for f in flags
    ]
    nearest_idx = np.argmin(distances)
    nearest_flag = flags[nearest_idx]
    nearest_dist_km = distances[nearest_idx] * 111  # approximate
    
    print("=" * 65)
    print("GEWS PoC Validation — Nepal 2026 Glacier-Rock Collapse")
    print("=" * 65)
    print()
    print(f"Known collapse site:  {SITE_LAT:.4f}°N, {SITE_LON:.4f}°E")
    print(f"Collapse date:        {EVENT_DATE}")
    print(f"Last satellite pass:  {ts_data['date_strings'][-1]}")
    print()
    print(f"Total flags detected: {len(flags)}")
    print(f"Nearest flag:         Flag {nearest_flag.flag_id}")
    print(f"  Location:           {nearest_flag.center_lat:.4f}°N, {nearest_flag.center_lon:.4f}°E")
    print(f"  Distance:           {nearest_dist_km:.1f} km from collapse site")
    print(f"  Anomaly score:      {nearest_flag.score:.2f}")
    print(f"  Peak z-score:       {nearest_flag.peak_zscore:.1f}")
    print(f"  Area:               {nearest_flag.area_m2:,.0f} m²")
    print(f"  Acceleration:       {nearest_flag.acceleration_m_yr2 * 1000:.2f} mm/yr²")
    
    if nearest_flag.voight_fit:
        print(f"  Voight R²:          {nearest_flag.voight_fit['r_squared']:.3f}")
        print(f"  Predicted failure:  {nearest_flag.voight_fit['days_until_failure']:.0f} days after last obs")
    
    print()
    
    # Rank of nearest flag
    rank = sorted(range(len(flags)), key=lambda i: flags[i].score, reverse=True).index(nearest_idx) + 1
    print(f"Rank of nearest flag: #{rank} of {len(flags)} (by anomaly score)")
    
    # Z-score at collapse site over time
    z_at_site = accel_map.acceleration_zscore[:, site_row, site_col]
    valid_z = z_at_site[np.isfinite(z_at_site)]
    if len(valid_z) > 0:
        n_above = np.sum(np.abs(valid_z) > 2.5)
        print(f"\nCollapse site z-score exceeded ±2.5σ in {n_above} of {len(valid_z)} windows")
        print(f"Maximum |z-score| at site: {np.max(np.abs(valid_z)):.1f}")
        
        # When did it first exceed threshold?
        exceedances = np.where(np.abs(z_at_site) > 2.5)[0]
        if len(exceedances) > 0:
            first_exc = exceedances[0]
            first_date = datetime.fromordinal(int(accel_map.window_centers[first_exc]))
            days_before = (datetime.strptime(EVENT_DATE, "%Y-%m-%d") - first_date).days
            print(f"First threshold exceedance: {first_date.strftime('%Y-%m-%d')} ({days_before} days before collapse)")
    
    if nearest_dist_km < 5:
        print("\n✓ DETECTION CONFIRMED — flag is within 5 km of known collapse site")
    elif nearest_dist_km < 15:
        print(f"\n◐ DETECTION PROXIMATE — flag is {nearest_dist_km:.1f} km from collapse site")
        print("  (within the 15 km study area; may be the same deforming system)")
    else:
        print(f"\n✗ NO DETECTION NEAR SITE — nearest flag is {nearest_dist_km:.1f} km away")
        print("  Consider adjusting detection parameters or verifying site coordinates")
else:
    print("No flags detected. Consider lowering sigma_threshold.")

---
## Next Steps

If the validation above confirms detection:

1. **Refine coordinates** — compare with post-event satellite imagery
   to pinpoint the actual collapse source. Update `SITE_LAT` / `SITE_LON`.

2. **Multi-burst analysis** — process additional bursts covering the
   full study area (not just one burst). This shows the spatial extent
   of the deforming zone.

3. **Process additional tracks** — Tracks 85 and 121 provide different
   viewing geometries. Combining them resolves the 3D displacement
   vector (not just line-of-sight).

4. **Validate against Chamoli 2021 and Aru 2016** — repeat this
   notebook for other documented collapses to test generalizability.

5. **Run Tier 1 cascade filtering** — add DEM and population data
   to assess downstream risk for each flagged site.

---
*GEWS v0.1.0 — Proof of Concept*